<a href="https://colab.research.google.com/github/amathie5/PPS-Project-/blob/main/PPS_final_part.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpBinary, value, LpStatus, PULP_CBC_CMD
import pandas as pd

Etape 1: créer un système de base (pas opti)
idée:
- deconposer chaque order en job individuel
- associer chaque job à un type (heritag ou Chrono) et aux paramètree sindividuel (realse date, due date, priotity)
- séquencer les tâches et le temps de traittement


In [ ]:
# ============================
# ETAPE 1 : CREATION DES JOBS
# ============================

# On commence par définir les ORDRES (données du problème)
orders = [
    {"id": 1, "H": 2, "C": 1, "r": 0,  "d": 40, "p": 2},
    {"id": 2, "H": 0, "C": 3, "r": 20, "d": 40, "p": 1},
    {"id": 3, "H": 1, "C": 0, "r": 10, "d": 30, "p": 3},
    {"id": 4, "H": 1, "C": 2, "r": 5,  "d": 40, "p": 1},
    {"id": 5, "H": 0, "C": 2, "r": 34, "d": 50, "p": 2},
]
orders_dict = {o["id"]: o for o in orders}

# H = Heritage, C = Chronos
# r = release date, d = due date, p = priorité


# ----------------------------
# Définition des routings
# ----------------------------

# Chaque type de montre a une gamme opératoire différente
routing_H = [
    ("Assembly", 3),
    ("Crystal", 1),
    ("Bracelet", 0.5),
    ("Inspection", 2)
]

routing_C = [
    ("Assembly", 4),
    ("Marker", 1.5),
    ("Crystal", 1),
    ("Inspection", 2.5)
]


# ----------------------------
# Création des jobs unitaires
# ----------------------------

jobs = []      # liste finale des jobs
job_id = 1     # compteur pour numéroter les jobs

# On parcourt chaque ordre
for o in orders:

    # 1) On crée les jobs HERITAGE
    for i in range(o["H"]):

        job = {
            "id": job_id,
            "order_id": o["id"],
            "type": "H",
            "release": o["r"],
            "due": o["d"],
            "priority": o["p"],
            "routing": routing_H
        }

        jobs.append(job)
        job_id += 1


    # 2) On crée les jobs CHRONOS
    for i in range(o["C"]):

        job = {
            "id": job_id,
            "order_id": o["id"],
            "type": "C",
            "release": o["r"],
            "due": o["d"],
            "priority": o["p"],
            "routing": routing_C
        }

        jobs.append(job)
        job_id += 1


# ----------------------------
# Vérification des résultats
# ----------------------------

print("Nombre total de jobs :", len(jobs))
print()

# On affiche quelques jobs pour vérifier
for j in jobs:
    print(j)

Nombre total de jobs : 12

{'id': 1, 'order_id': 1, 'type': 'H', 'release': 0, 'due': 40, 'priority': 2, 'routing': [('Assembly', 3), ('Crystal', 1), ('Bracelet', 0.5), ('Inspection', 2)]}
{'id': 2, 'order_id': 1, 'type': 'H', 'release': 0, 'due': 40, 'priority': 2, 'routing': [('Assembly', 3), ('Crystal', 1), ('Bracelet', 0.5), ('Inspection', 2)]}
{'id': 3, 'order_id': 1, 'type': 'C', 'release': 0, 'due': 40, 'priority': 2, 'routing': [('Assembly', 4), ('Marker', 1.5), ('Crystal', 1), ('Inspection', 2.5)]}
{'id': 4, 'order_id': 2, 'type': 'C', 'release': 20, 'due': 40, 'priority': 1, 'routing': [('Assembly', 4), ('Marker', 1.5), ('Crystal', 1), ('Inspection', 2.5)]}
{'id': 5, 'order_id': 2, 'type': 'C', 'release': 20, 'due': 40, 'priority': 1, 'routing': [('Assembly', 4), ('Marker', 1.5), ('Crystal', 1), ('Inspection', 2.5)]}
{'id': 6, 'order_id': 2, 'type': 'C', 'release': 20, 'due': 40, 'priority': 1, 'routing': [('Assembly', 4), ('Marker', 1.5), ('Crystal', 1), ('Inspection', 2.5)]

cela nous donne liste complète des jobs, le nrouting, et les paramètres


Phase 2 heuristiques:
Heuristics are introduced in the scheduling phase (step 2), where sequencing decisions must be made under capacity constraints.

In [ ]:
# ============================
# ETAPE 2 : HEURISTIQUE DE SCHEDULING
# ============================

# Hypothèse : la liste "jobs" vient de l'étape 1

# ----------------------------
# 1) Trier les jobs (règle heuristique)
# ----------------------------

# Tri par due date puis priorité
jobs_sorted = sorted(jobs, key=lambda j: (j["due"], j["priority"]))

print("Ordre des jobs (heuristique EDD + priorité) :")
for j in jobs_sorted:
    print(f"Job {j['id']} | due={j['due']} | priority={j['priority']}")


# ----------------------------
# 2) Initialisation des machines
# ----------------------------

# Temps de disponibilité de chaque station
machines = {
    "Assembly_1": 0,
    "Assembly_2": 0,
    "Crystal": 0,
    "Bracelet": 0,
    "Marker": 0,
    "Inspection": 0
}

# On stocke le planning final
schedule = []


# ----------------------------
# 3) Construction du planning
# ----------------------------

for job in jobs_sorted:

    current_time = job["release"]  # on respecte la release date

    # On parcourt les opérations du routing
    for (machine_type, proc_time) in job["routing"]:

        # Cas spécial : Assembly a 2 machines → on prend la plus tôt disponible
        if machine_type == "Assembly":
            m1 = machines["Assembly_1"]
            m2 = machines["Assembly_2"]

            if m1 <= m2:
                machine = "Assembly_1"
            else:
                machine = "Assembly_2"
        else:
            machine = machine_type

        # Calcul du start time (ASAP)
        start = max(current_time, machines[machine])
        end = start + proc_time

        # Mise à jour
        machines[machine] = end
        current_time = end

        # Stockage
        schedule.append({
            "job": job["id"],
            "machine": machine,
            "start": start,
            "end": end
        })


# ----------------------------
# 4) Affichage du planning
# ----------------------------

print("\nPlanning généré :\n")

for s in schedule:
    print(f"Job {s['job']} | {s['machine']} | {s['start']} -> {s['end']}")

Ordre des jobs (heuristique EDD + priorité) :
Job 7 | due=30 | priority=3
Job 4 | due=40 | priority=1
Job 5 | due=40 | priority=1
Job 6 | due=40 | priority=1
Job 8 | due=40 | priority=1
Job 9 | due=40 | priority=1
Job 10 | due=40 | priority=1
Job 1 | due=40 | priority=2
Job 2 | due=40 | priority=2
Job 3 | due=40 | priority=2
Job 11 | due=50 | priority=2
Job 12 | due=50 | priority=2

Planning généré :

Job 7 | Assembly_1 | 10 -> 13
Job 7 | Crystal | 13 -> 14
Job 7 | Bracelet | 14 -> 14.5
Job 7 | Inspection | 14.5 -> 16.5
Job 4 | Assembly_2 | 20 -> 24
Job 4 | Marker | 24 -> 25.5
Job 4 | Crystal | 25.5 -> 26.5
Job 4 | Inspection | 26.5 -> 29.0
Job 5 | Assembly_1 | 20 -> 24
Job 5 | Marker | 25.5 -> 27.0
Job 5 | Crystal | 27.0 -> 28.0
Job 5 | Inspection | 29.0 -> 31.5
Job 6 | Assembly_1 | 24 -> 28
Job 6 | Marker | 28 -> 29.5
Job 6 | Crystal | 29.5 -> 30.5
Job 6 | Inspection | 31.5 -> 34.0
Job 8 | Assembly_2 | 24 -> 27
Job 8 | Crystal | 30.5 -> 31.5
Job 8 | Bracelet | 31.5 -> 32.0
Job 8 | In

In [ ]:
# ============================
# ETAPE 2 : HEURISTIQUE DE SCHEDULING
# ============================

# ----------------------------
# 1) Trier les jobs (règle heuristique)
# ----------------------------

jobs_sorted = sorted(jobs, key=lambda j: (j["due"], j["priority"]))

print("Ordre des jobs (heuristique EDD + priorité) :")
for j in jobs_sorted:
    print(f"Job {j['id']} | Order {j['order_id']} | due={j['due']} | priority={j['priority']}")


# ----------------------------
# 2) Initialisation des machines
# ----------------------------

machines = {
    "Assembly_1": 0,
    "Assembly_2": 0,
    "Crystal": 0,
    "Bracelet": 0,
    "Marker": 0,
    "Inspection": 0
}

schedule = []

# 🔥 NOUVEAU : suivi des commandes
order_completion = {}


# ----------------------------
# 3) Construction du planning
# ----------------------------

for job in jobs_sorted:

    current_time = job["release"]

    for (machine_type, proc_time) in job["routing"]:

        if machine_type == "Assembly":
            m1 = machines["Assembly_1"]
            m2 = machines["Assembly_2"]

            if m1 <= m2:
                machine = "Assembly_1"
            else:
                machine = "Assembly_2"
        else:
            machine = machine_type

        start = max(current_time, machines[machine])
        end = start + proc_time

        machines[machine] = end
        current_time = end

        schedule.append({
            "job": job["id"],
            "order": job["order_id"],   # 🔥 lien visible dans le planning
            "machine": machine,
            "start": start,
            "end": end
        })

    # 🔥 NOUVEAU : fin du job
    job["completion"] = current_time

    # 🔥 NOUVEAU : mise à jour de la commande
    o = job["order_id"]

    if o not in order_completion:
        order_completion[o] = 0

    order_completion[o] = max(order_completion[o], job["completion"])


# ----------------------------
# 4) Calcul du retard par commande
# ----------------------------

order_tardiness = {}

for job in jobs:
    o = job["order_id"]
    due = job["due"]

    order_tardiness[o] = max(0, order_completion[o] - due)


# ----------------------------
# 5) Affichage
# ----------------------------

print("\nPlanning généré :\n")

for s in schedule:
    print(f"Job {s['job']} (Order {s['order']}) | {s['machine']} | {s['start']} -> {s['end']}")

print("\nCompletion par commande :")
for o, c in order_completion.items():
    print(f"Order {o} terminé à t = {c}")

print("\nRetard par commande :")
for o, t in order_tardiness.items():
    print(f"Order {o} retard = {t}")

Ordre des jobs (heuristique EDD + priorité) :
Job 7 | Order 3 | due=30 | priority=3
Job 4 | Order 2 | due=40 | priority=1
Job 5 | Order 2 | due=40 | priority=1
Job 6 | Order 2 | due=40 | priority=1
Job 8 | Order 4 | due=40 | priority=1
Job 9 | Order 4 | due=40 | priority=1
Job 10 | Order 4 | due=40 | priority=1
Job 1 | Order 1 | due=40 | priority=2
Job 2 | Order 1 | due=40 | priority=2
Job 3 | Order 1 | due=40 | priority=2
Job 11 | Order 5 | due=50 | priority=2
Job 12 | Order 5 | due=50 | priority=2

Planning généré :

Job 7 (Order 3) | Assembly_1 | 10 -> 13
Job 7 (Order 3) | Crystal | 13 -> 14
Job 7 (Order 3) | Bracelet | 14 -> 14.5
Job 7 (Order 3) | Inspection | 14.5 -> 16.5
Job 4 (Order 2) | Assembly_2 | 20 -> 24
Job 4 (Order 2) | Marker | 24 -> 25.5
Job 4 (Order 2) | Crystal | 25.5 -> 26.5
Job 4 (Order 2) | Inspection | 26.5 -> 29.0
Job 5 (Order 2) | Assembly_1 | 20 -> 24
Job 5 (Order 2) | Marker | 25.5 -> 27.0
Job 5 (Order 2) | Crystal | 27.0 -> 28.0
Job 5 (Order 2) | Inspection |

Étape 1 – Structuration des données
Les ordres sont décomposés en jobs individuels. Chaque job hérite des informations de sa commande (release date, due date, priorité) et se voit attribuer un routing spécifique selon le type de montre.
Étape 2 – Définition de l’heuristique
Une règle de priorité est choisie pour ordonner les jobs. Ici, une heuristique combinant les due dates (EDD) et les priorités est utilisée afin de favoriser les commandes urgentes.
Étape 3 – Initialisation du système
Les ressources (machines) sont initialisées avec leurs disponibilités. Une structure de stockage est également créée pour enregistrer le planning et suivre l’avancement des commandes.
Étape 4 – Construction du planning
Les jobs sont planifiés séquentiellement selon l’heuristique. Chaque opération est affectée à une machine en respectant les contraintes de précédence, de disponibilité des ressources et les release dates. Le planning est construit au plus tôt (ASAP).
Étape 5 – Évaluation des performances
Les temps de complétion des jobs sont agrégés au niveau des commandes. Le retard (tardiness) est calculé pour chaque ordre, permettant d’évaluer la qualité du planning obtenu.

